# Drone Flyby: synthetic data + YOLO training (Kaggle) - v4

Settings (right-hand panel): **Accelerator = GPU T4 x2**, **Internet = on**.
Add via *Add Input*: our `drone-flyby-code` dataset, plus the aerial photo datasets
`sagar100rathod/inria-aerial-image-labeling-dataset` (cities) and
`adrianboguszewski/landcoverai` (countryside, water). They are the backgrounds the
objects get pasted onto, so the model stops mistaking boats and bushes for objects.
Run the cells top to bottom. At the end, download `best.pt` from the Output tab.

v4 starts from v3 (`drone-yolo11n-v3.pt`, inside the code zip), pastes 600 cut-outs of 24 objects from the recorded validation flight (`training/patches_val`), pastes weak classes more often and uses fewer Helsinki backgrounds.

In [ ]:
# Same ultralytics version as the laptop, so best.pt loads in the service.
!pip install -q ultralytics==8.4.152

In [ ]:
# Official repo: the 25 Helsinki frames + utils.py/dtos.py the scripts import.
!git clone --depth 1 https://github.com/amboltio/Nordic-AI-Cup-2026.git /tmp/official
import glob, shutil
src = glob.glob('/kaggle/input/**/training/make_dataset.py', recursive=True)
assert src, 'Add the drone-flyby-code dataset as input'
shutil.copytree(src[0].rsplit('/', 1)[0], '/tmp/official/drone-flyby/training', dirs_exist_ok=True)
%cd /tmp/official/drone-flyby

In [ ]:
# 1. Cut the objects out.  2. Build the dataset on many backgrounds (~20 min).
%env OPENCV_LOG_LEVEL=ERROR
!python training/extract_patches.py
!python training/make_dataset.py --scenes 1600 --out /tmp/yolo --backgrounds /kaggle/input --extra-patches training/patches_val --helsinki-share 0.1 --class-weights small_plane=2.5,helicopter=2,large_tower=2.5,medium_plane=2.5,medium_launcher=2.5,jammer=2,large_launcher=2,tank=1.5,spacecraft=1.5,small_launcher=1.5,ta-ta=1.5,condor=1.5


In [ ]:
from ultralytics import YOLO

# Start from v3 (same classes, same size).
model = YOLO('training/drone-yolo11n-v3.pt')
model.train(
    data='/tmp/yolo/data.yaml',
    imgsz=960,          # views are 960x540: no extra shrinking
    epochs=40,
    batch=16,
    device=0,
    cache='ram',        # PNG decoding is the bottleneck otherwise
    workers=4,
    # The drone flies at a fixed altitude, so objects barely change size.
    scale=0.1,
    degrees=0.0,        # rotation already done while pasting
    flipud=0.5,
    fliplr=0.5,
    project='/kaggle/working/runs',
    name='yolo11n_960_v4',
    plots=True,
)

In [ ]:
import shutil
shutil.copy('/kaggle/working/runs/yolo11n_960_v4/weights/best.pt', '/kaggle/working/drone-yolo11n-v4.pt')
print('Download drone-yolo11n-v4.pt from the Output tab')